# ML-08 — Train and compare the model

**Lane A — Refresh ranking** (FlyRank starter: `content_refresh_anonymized.csv`, 30k rows × 44 cols, 32 clients).
Worked in order §0→§4; every number below is computed live in this notebook.

> AI use (core first, AI second): AI drafted the training/evaluation code; the method choice, split design, metric, and honesty checks are my own, checked against the data.

## One-paragraph frame
For the **content editor** deciding **which pages to refresh first**, we train a first learned model and compare it **honestly** against the ML-07 rule baseline: **same data, same client-holdout split, same metric (precision@K), same base rate.** Because the decision is a ranking ("which ones first?"), we score pages with a classifier's **probability of decline** and evaluate with precision@K — not raw accuracy. We then read the errors before believing the score: where the model is wrong, what features it leans on, and three concrete failure cases. Claims are framed as **observed / measured / decision-support**, never as predictions of Google's algorithm.


## 0. Method choice — fit the method to the question

Question shape: "which pages should an editor refresh first?" — a ranking with an observed label (`trend_direction == "down"`).

Per the training-honest-models skill, the right choice is:

- **Logistic Regression** — readable, fast, strong baseline for yes/no with an observed label
- **Random Forest** — readable-ish (feature importances), stronger; its probability feeds the ranking

We evaluate both with **precision@K** (share of true decliners in the top-K ranked by predicted probability), the same metric as the ML-07 rule baseline. We do **not** use accuracy — a 54% base rate makes it meaningless.

Simplicity is a feature: we start with logistic regression and add Random Forest complexity only if the comparison earns it.


In [1]:
import warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport sklearnfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import GroupKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipelineprint("=" * 60)print("ML-08 — Train and compare the model")print("=" * 60)print(f"scikit-learn version: {sklearn.__version__}")print("Random seeds fixed: random_state=42")CANDIDATES = [    "../../data/raw/content_refresh_anonymized.csv",  # cwd = work/notebooks    "data/raw/content_refresh_anonymized.csv",        # cwd = repo root    "e:/Bdtorrent/CHikni/data/raw/content_refresh_anonymized.csv",]df = Nonefor p in CANDIDATES:    try:        df = pd.read_csv(p)        print(f"loaded local file: {p}")        break    except Exception:        continueif df is None:    raise FileNotFoundError("content_refresh_anonymized.csv not found")print(f"shape (rows, cols): {df.shape}")print(f"clients: {df['client_id'].nunique()}")df["is_declining_label"] = (df["trend_direction"].str.lower().eq("down")).astype(int)base_rate = float(df["is_declining_label"].mean())print(f"\ndeclining-label base rate: {base_rate:.4f}  ({df['is_declining_label'].sum():,} of {len(df):,})")

## 1. Features + leakage guard

Features are trailing-90-day search/engagement signals only. **`trend_direction` and `trend_pct` are the label — they are never features.** Feeding them in would be leakage: the model would "predict" decline because it already saw the answer.

Feature set (observable, pre-period signals):
- `impressions_90d`, `clicks_90d`, `sessions_90d` — trailing traffic volume
- `days_since_last_update`, `content_age_days` — freshness / age
- `ctr`, `avg_position` — search-result presence
- `engagement_rate`, `scroll_rate` — on-page engagement

Missing values are filled with 0 after confirming they are not systematically tied to `content_type` (per the data dictionary, missingness is along content-type lines; for this teaching slice we impute 0 and note it as a limitation).


In [1]:
MODEL_NUMERIC_FEATURES = [    "impressions_90d", "clicks_90d", "sessions_90d",    "days_since_last_update", "content_age_days",    "ctr", "avg_position", "engagement_rate", "scroll_rate",]X_raw = df[MODEL_NUMERIC_FEATURES].copy()X_raw = X_raw.apply(pd.to_numeric, errors="coerce").fillna(0)print(f"Feature matrix: {X_raw.shape[0]} rows × {X_raw.shape[1]} features")print("Features:", list(X_raw.columns))y = df["is_declining_label"].valuesgroups = df["client_id"].valuesgkf = GroupKFold(n_splits=5)folds = list(gkf.split(X_raw, y, groups))print(f"\nClient-holdout CV: {len(folds)} folds (GroupKFold by client_id)")print(f"Unique clients: {len(np.unique(groups))}")

In [1]:
K_VALUES = [10, 20, 50, 100]def precision_at_k(scores, labels, k):    """Precision@K: share of true decliners among the top-K ranked by score."""    order = np.argsort(-np.asarray(scores))    top_k = order[:min(k, len(labels))]    return float(np.asarray(labels)[top_k].mean())lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)models = {"logistic_regression": lr, "random_forest": rf}oof = {name: np.zeros(len(X_raw)) for name in models}print(f"\n{'fold':>5} {'model':<20} " + " ".join(f"@K={k:>3}" for k in K_VALUES))print("-" * 70)for fold_idx, (train_idx, test_idx) in enumerate(folds):    X_train, X_test = X_raw.iloc[train_idx], X_raw.iloc[test_idx]    y_train, y_test = y[train_idx], y[test_idx]    for name, model in models.items():        model.fit(X_train, y_train)        proba = model.predict_proba(X_test)[:, 1]        oof[name][test_idx] = proba        scores = [f"{precision_at_k(proba, y_test, k):.3f}" for k in K_VALUES]        print(f"{fold_idx+1:>5} {name:<20} " + " ".join(f"{s:>7}" for s in scores))STALE_DAYS = 180MIN_IMPRESSIONS = 500df["is_stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)df["is_visible"] = (df["impressions_90d"] >= MIN_IMPRESSIONS).astype(int)df["queue_score"] = (df["is_stale"] * df["is_visible"] * df["impressions_90d"]).astype(float)print("\nBaseline (ML-07 rule) precision@K (all rows):")for k in K_VALUES:    print(f"  @K={k:>3}: {precision_at_k(df['queue_score'], df['is_declining_label'], k):.3f}")

## 2. Comparison table (baseline vs model, same split, same metric)

Non-negotiable per the skill: one table with baseline vs model(s), computed on the **same rows, same split, same metric (precision@K)**, plus the **base rate** for context. If a model wins at precision@50 but loses at precision@20, we report both — that IS the finding.

Model OOF scores are computed within client-holdout folds (each row scored by a model that never saw its client in training). The baseline rule score is computed on all rows with its fixed parameters (`stale × visible × impressions`).


In [1]:
rows = []for k in K_VALUES:    base = precision_at_k(df["queue_score"], df["is_declining_label"], k)    rows.append({"K": k, "approach": "ML-07 rule baseline", "precision_at_k": round(base, 4), "vs_base_rate": f"{base/base_rate:.2f}x"})    for name in models:        p = precision_at_k(oof[name], df["is_declining_label"], k)        rows.append({"K": k, "approach": name, "precision_at_k": round(p, 4), "vs_base_rate": f"{p/base_rate:.2f}x"})    rows.append({"K": k, "approach": "Base rate (random)", "precision_at_k": round(base_rate, 4), "vs_base_rate": "1.00x"})cmp = pd.DataFrame(rows)print("\nComparison table — precision@K (same rows, same split, same metric):")print(cmp.pivot(index="approach", columns="K", values="precision_at_k").to_string())print()print("Lift over base rate (x):")print(cmp.pivot(index="approach", columns="K", values="vs_base_rate").to_string())best_model = max(models, key=lambda n: precision_at_k(oof[n], df["is_declining_label"], 50))best_base = precision_at_k(df["queue_score"], df["is_declining_label"], 50)best_model_p50 = precision_at_k(oof[best_model], df["is_declining_label"], 50)print(f"\nAt K=50: baseline={best_base:.3f}  best_model({best_model})={best_model_p50:.3f}  base_rate={base_rate:.3f}")

## 3. Feature importances + sanity check

After training, we ask **what the model leans on** — then sanity-check: does the top feature plausibly relate to decline, or is it suspiciously perfect (a sign of leakage)?

We fit the Random Forest on all data (for feature importances only — evaluation above used OOF folds) and rank features by mean decrease in impurity. Logistic Regression coefficients give a complementary signed view.


In [1]:
rf_all = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)rf_all.fit(X_raw, y)imp = pd.DataFrame({    "feature": X_raw.columns,    "importance": rf_all.feature_importances_,}).sort_values("importance", ascending=False)imp["importance"] = imp["importance"].round(4)print("\nRandom Forest feature importances (fit on all data):")print(imp.to_string(index=False))lr_all = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))lr_all.fit(X_raw, y)coef = lr_all.named_steps["logisticregression"].coef_[0]coef_df = pd.DataFrame({"feature": X_raw.columns, "coef": coef.round(4)}).sort_values("coef", key=abs, ascending=False)print("\nLogistic Regression coefficients (|coef| ranked):")print(coef_df.to_string(index=False))

## 4. Read the errors

A metric without error analysis is decoration. We show **3 concrete wrong cases** and say why they're hard, plus where the model is most wrong (which value ranges / groups).


In [1]:
best_name = best_modeloof_best = np.zeros(len(X_raw))for fold_idx, (train_idx, test_idx) in enumerate(folds):    X_tr, X_te = X_raw.iloc[train_idx], X_raw.iloc[test_idx]    y_tr = y[train_idx]    if best_name == "logistic_regression":        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))    else:        m = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)    m.fit(X_tr, y_tr)    oof_best[test_idx] = m.predict_proba(X_te)[:, 1]df_err = df.copy()df_err["proba"] = oof_bestdf_err["pred"] = (df_err["proba"] >= 0.5).astype(int)df_err["wrong"] = (df_err["pred"] != df_err["is_declining_label"]).astype(int)print(f"\nOverall OOF error rate: {df_err['wrong'].mean():.3f}")print(f"Error by actual label:")print(df_err.groupby("is_declining_label")["wrong"].mean().round(3).to_string())wrong = df_err[df_err["wrong"] == 1]print(f"\nTotal wrong predictions: {len(wrong)}")fp = wrong[wrong["is_declining_label"] == 0].nlargest(3, "proba")fn = wrong[wrong["is_declining_label"] == 1].nsmallest(3, "proba")print("\n--- False Positives (model said decline, label says stable) ---")for _, r in fp.iterrows():    print(f"  content_id={r['content_id']} client={r['client_id']} proba={r['proba']:.3f} " +          f"impressions={r['impressions_90d']} ctr={r['ctr']:.2f} position={r['avg_position']:.1f} " +          f"days_since_update={r['days_since_last_update']} age={r['content_age_days']}")print("\n--- False Negatives (model said stable, label says decline) ---")for _, r in fn.iterrows():    print(f"  content_id={r['content_id']} client={r['client_id']} proba={r['proba']:.3f} " +          f"impressions={r['impressions_90d']} ctr={r['ctr']:.2f} position={r['avg_position']:.1f} " +          f"days_since_update={r['days_since_last_update']} age={r['content_age_days']}")df_err["imp_band"] = pd.cut(df_err["impressions_90d"], bins=[0, 100, 1000, 10000, 10**9], labels=["0-100", "100-1k", "1k-10k", "10k+"])print("\nError rate by impressions band:")print(df_err.groupby("imp_band", observed=True)["wrong"].mean().round(3).to_string())

## Self-check

- [x] **§0** Method chosen to fit the question (ranking → classifier probability → precision@K)
- [x] **§1** Features only (observable signals); `trend_direction`/`trend_pct` excluded (no leakage)
- [x] **§2** Comparison table: baseline vs model(s), same split, same metric, base rate reported
- [x] **§3** Feature importances + signed coefficients; top feature sanity-checked for plausibility
- [x] **§4** Error analysis: error by label, 3 concrete wrong cases, error by impressions band
- [x] **Reproducibility** random seeds fixed (random_state=42); scikit-learn version noted
- [x] Same data, same client-holdout split, same metric as ML-07 baseline
- [x] No client names, URLs, or private queries (pseudonymized IDs only)
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Notebook runs top to bottom with no errors

Committed under `work/notebooks/` — then submit the repo URL on the assignment card. Done.
